# From Scratch to Fine-Tuned: The CLIP Training Journey

This notebook walks through what our model learned at each stage,
using the same tasks so you can see the improvement directly.

**Three models compared side-by-side:**
1. **From scratch** (exp 003) — our best result training from random init on 1M CC3M pairs
2. **Pretrained** (OpenAI) — zero-shot, no fine-tuning, trained on 400M pairs
3. **Fine-tuned** (exp 007) — pretrained model adapted on our CC3M data

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image
from torch.amp import autocast
from pathlib import Path
from collections import OrderedDict

from src.model import create_model
from src.dataset import CC3MDataset, create_dataloader
from src.eval import compute_recall_at_k

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda')
random.seed(42)
torch.manual_seed(42)

## Load all three models

In [ ]:
# ---- CONFIGURE CHECKPOINT PATHS ----
SCRATCH_CKPT = '../experiments/003_lower_lr/checkpoints/step_020000.pt'
FINETUNE_CKPT = '../experiments/007_finetune/checkpoints_c/step_005000.pt'

models = OrderedDict()

# 1. From scratch (exp 003)
m, preprocess, tokenizer = create_model('ViT-B-32', device=device)
ckpt = torch.load(SCRATCH_CKPT, map_location=device, weights_only=False)
m.load_state_dict(ckpt['model_state_dict'])
m.eval()
models['From Scratch\n(exp 003, 1M pairs)'] = m
print(f'Loaded from-scratch model (step {ckpt["step"]})')

# 2. Pretrained (OpenAI, zero-shot)
m2, _, _ = create_model('ViT-B-32', pretrained='openai', device=device)
m2.eval()
models['Pretrained\n(OpenAI, 400M pairs)'] = m2
print('Loaded pretrained model')

# 3. Fine-tuned (exp 007, lr=1e-5)
m3, _, _ = create_model('ViT-B-32', device=device)
ckpt3 = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
m3.load_state_dict(ckpt3['model_state_dict'])
m3.eval()
models['Fine-tuned\n(exp 007, lr=1e-5)'] = m3
print(f'Loaded fine-tuned model (step {ckpt3["step"]})')

In [ ]:
# Load eval dataset
dataset = CC3MDataset(
    tsv_path='../data/cc3m/Validation_GCC-1.1.0-Validation.tsv',
    image_dir='../data/eval/images',
    transform=preprocess,
    tokenizer=tokenizer,
)
print(f'{len(dataset)} eval images')

# Helpers
@torch.no_grad()
def encode_images(model, paths):
    imgs = torch.stack([preprocess(Image.open(p).convert('RGB')) for p in paths]).to(device)
    with autocast('cuda', dtype=torch.bfloat16):
        return F.normalize(model.encode_image(imgs), dim=-1)

@torch.no_grad()
def encode_texts(model, texts):
    tokens = tokenizer(texts).to(device)
    with autocast('cuda', dtype=torch.bfloat16):
        return F.normalize(model.encode_text(tokens), dim=-1)

---
## 1. Recall@K — The headline metric

Given an image, can the model find its matching caption among 1000 candidates?
This is the most objective comparison we can make.

In [ ]:
subset = torch.utils.data.Subset(dataset, list(range(1000)))
loader = create_dataloader(subset, batch_size=256, num_workers=4, shuffle=False)

all_results = {}
for name, model in models.items():
    results = compute_recall_at_k(model, loader)
    all_results[name] = results
    short = name.split('\n')[0]
    print(f'{short:15s}  R@1={results["i2t_recall@1"]:.3f}  R@5={results["i2t_recall@5"]:.3f}')

# Plot
metrics = ['i2t_recall@1', 't2i_recall@1', 'i2t_recall@5', 't2i_recall@5']
labels = ['Image→Text R@1', 'Text→Image R@1', 'Image→Text R@5', 'Text→Image R@5']
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, results) in enumerate(all_results.items()):
    vals = [results[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name.replace('\n', ' '), color=colors[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.1%}', ha='center', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_ylabel('Recall')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.set_title('Retrieval Performance: From Scratch → Pretrained → Fine-tuned', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Zero-shot classification — side by side

Give the same image to all three models. Which labels do they pick?
Run this cell multiple times to see different images.

In [ ]:
# ---- CHANGE THESE ----
idx = random.randint(0, len(dataset) - 1)
labels = [
    'a dog', 'a cat', 'a car', 'food', 'a building',
    'a person', 'nature', 'art', 'sports', 'water',
]

caption, img_path = dataset.samples[idx]
prompts = [f'a photo of {l}' for l in labels]

fig, axes = plt.subplots(1, 4, figsize=(22, 6),
                         gridspec_kw={'width_ratios': [1, 1, 1, 1]})

# Image
axes[0].imshow(Image.open(img_path).convert('RGB'))
axes[0].set_title(f'Caption:\n"{caption[:60]}"', fontsize=9)
axes[0].axis('off')

# Classification for each model
for ax, (name, model) in zip(axes[1:], models.items()):
    img_feat = encode_images(model, [img_path])
    txt_feat = encode_texts(model, prompts)
    probs = (img_feat @ txt_feat.t()).squeeze(0).float().softmax(dim=0).cpu().numpy()

    sorted_idx = np.argsort(probs)
    colors_bar = ['#2ecc71' if i == sorted_idx[-1] else '#bdc3c7' for i in range(len(labels))]
    ax.barh([labels[i] for i in sorted_idx], [probs[i] for i in sorted_idx],
            color=[colors_bar[i] for i in sorted_idx])
    for i, idx_s in enumerate(sorted_idx):
        ax.text(probs[idx_s] + 0.005, i, f'{probs[idx_s]:.0%}', va='center', fontsize=8)
    ax.set_xlim(0, max(probs) * 1.3)
    ax.set_title(name, fontsize=10, fontweight='bold')

fig.suptitle('Zero-Shot Classification Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Text → Image retrieval — same query, three models

Type a query. Each model searches the same pool of 500 images.
Which model finds the best matches?

In [ ]:
# ---- CHANGE THIS ----
query = "a dog playing in the park"
TOP_K = 5

# Build a shared image pool
random.seed(42)
pool_idx = random.sample(range(len(dataset)), 500)
pool_paths = [dataset.samples[i][1] for i in pool_idx]

# Encode pool with each model
pool_features = {}
for name, model in models.items():
    feats = []
    for i in range(0, len(pool_paths), 64):
        batch = pool_paths[i:i+64]
        valid = []
        for p in batch:
            try:
                Image.open(p).verify()
                valid.append(p)
            except: continue
        if valid:
            feats.append(encode_images(model, valid))
    pool_features[name] = torch.cat(feats, dim=0)

# Query each model
n_models = len(models)
fig, axes = plt.subplots(n_models, TOP_K, figsize=(4 * TOP_K, 5 * n_models))

for row, (name, model) in enumerate(models.items()):
    txt_feat = encode_texts(model, [query])
    sims = (txt_feat @ pool_features[name].t()).squeeze(0).float()
    topk_vals, topk_idx = sims.topk(TOP_K)

    for col, (sim, idx) in enumerate(zip(topk_vals, topk_idx)):
        ax = axes[row][col]
        img = Image.open(pool_paths[idx.item()]).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'sim={sim.item():.3f}', fontsize=9)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(name, fontsize=10, fontweight='bold',
                         rotation=0, labelpad=100, va='center')

fig.suptitle(f'Text → Image Retrieval: "{query}"', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Similarity heatmap — matching accuracy

8 random image-caption pairs. The diagonal should be the brightest in each
row — that means the model correctly matches each image to its caption.
Compare how sharp the diagonal is across models.

In [ ]:
N = 8
random.seed(77)
sample_idx = random.sample(range(len(dataset)), N)

images, captions, paths = [], [], []
for idx in sample_idx:
    try:
        img_tensor, _ = dataset[idx]
        images.append(img_tensor)
        captions.append(dataset.samples[idx][0][:35])
        paths.append(dataset.samples[idx][1])
    except: continue

n = len(images)
img_batch = torch.stack(images).to(device)
txt_tokens = tokenizer(captions).to(device)

fig = plt.figure(figsize=(22, 8))
gs = gridspec.GridSpec(2, len(models), height_ratios=[0.3, 1], hspace=0.4)

# Thumbnails across the top of each column
for col in range(len(models)):
    ax_imgs = fig.add_subplot(gs[0, col])
    # Create a strip of thumbnails
    strip = np.concatenate([np.array(Image.open(p).convert('RGB').resize((80, 80))) for p in paths], axis=1)
    ax_imgs.imshow(strip)
    ax_imgs.axis('off')

scores = []
for col, (name, model) in enumerate(models.items()):
    with torch.no_grad(), autocast('cuda', dtype=torch.bfloat16):
        img_feat = F.normalize(model.encode_image(img_batch), dim=-1)
        txt_feat = F.normalize(model.encode_text(txt_tokens), dim=-1)
    sim = (img_feat @ txt_feat.t()).float().cpu().numpy()

    correct = sum(sim[i].argmax() == i for i in range(n))
    scores.append(correct)

    ax = fig.add_subplot(gs[1, col])
    im = ax.imshow(sim, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    for i in range(n):
        for j in range(n):
            weight = 'bold' if i == j else 'normal'
            color = 'white' if sim[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                    fontsize=7, color=color, fontweight=weight)

    ax.set_xticks(range(n))
    ax.set_xticklabels([f't{i}' for i in range(n)], fontsize=7)
    ax.set_yticks(range(n))
    ax.set_yticklabels([f'i{i}' for i in range(n)], fontsize=7)
    ax.set_title(f'{name}\n{correct}/{n} correct', fontsize=10, fontweight='bold')

fig.suptitle('Image × Caption Similarity Matrix — Can the model match pairs?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Embedding geometry — how the models organize the world

Same 300 images projected into 2D with UMAP. Does the model put
similar things near each other?

In [ ]:
import umap
import re

CATEGORIES = {
    'person': ['person', 'people', 'man', 'woman', 'boy', 'girl', 'player', 'crowd', 'portrait'],
    'animal': ['dog', 'cat', 'bird', 'horse', 'fish', 'animal', 'deer', 'bear', 'pet', 'wildlife'],
    'vehicle': ['car', 'truck', 'bus', 'train', 'airplane', 'boat', 'bicycle', 'motorcycle'],
    'nature': ['sunset', 'mountain', 'ocean', 'beach', 'forest', 'tree', 'flower', 'lake', 'sky'],
    'building': ['building', 'house', 'church', 'castle', 'tower', 'bridge', 'hotel', 'temple'],
    'food': ['food', 'meal', 'plate', 'cake', 'pizza', 'fruit', 'coffee', 'restaurant'],
}

def categorize(caption):
    cl = caption.lower()
    for cat, kws in CATEGORIES.items():
        for kw in kws:
            if re.search(rf'\b{kw}\b', cl): return cat
    return 'other'

N_SAMPLES = 300
random.seed(42)
indices = random.sample(range(len(dataset)), N_SAMPLES)

# Encode with each model
all_feats = {}
cats = []
valid_paths = []
built_data = False

for name, model in models.items():
    feats = []
    for start in range(0, len(indices), 64):
        batch_idx = indices[start:start+64]
        imgs = []
        for idx in batch_idx:
            try:
                img_tensor, _ = dataset[idx]
                imgs.append(img_tensor)
                if not built_data:
                    cats.append(categorize(dataset.samples[idx][0]))
                    valid_paths.append(idx)
            except: continue
        if imgs:
            batch = torch.stack(imgs).to(device)
            with torch.no_grad(), autocast('cuda', dtype=torch.bfloat16):
                feats.append(F.normalize(model.encode_image(batch), dim=-1).cpu())
    all_feats[name] = torch.cat(feats, dim=0).float().numpy()
    built_data = True

print(f'Encoded {len(cats)} images. Running UMAP...')

# UMAP each model
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
unique_cats = sorted(set(cats))
cmap = plt.colormaps['tab10']
cat_colors = {c: cmap(i) for i, c in enumerate(unique_cats)}

for ax, (name, feats) in zip(axes, all_feats.items()):
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
    emb = reducer.fit_transform(feats)

    for cat in unique_cats:
        mask = [c == cat for c in cats]
        ax.scatter(emb[mask, 0], emb[mask, 1], c=[cat_colors[cat]],
                   alpha=0.6, s=15, label=cat)

    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

axes[-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
fig.suptitle('Embedding Space: How each model organizes visual concepts',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. The confidence gap

How confident is each model when it's right vs wrong?
A good model has high similarity for correct pairs and low for incorrect ones.
The gap between the two distributions is the model's "decision margin".

In [ ]:
# Use the same 1K eval subset
n_eval = 500
eval_imgs, eval_txts = [], []
for i in range(n_eval):
    try:
        img, txt = dataset[i]
        eval_imgs.append(img)
        eval_txts.append(txt)
    except: continue

img_batch = torch.stack(eval_imgs).to(device)
txt_batch = torch.stack(eval_txts).to(device)
n = len(eval_imgs)

fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)

for ax, (name, model) in zip(axes, models.items()):
    # Encode in chunks
    all_img_f, all_txt_f = [], []
    for start in range(0, n, 64):
        ib = img_batch[start:start+64]
        tb = txt_batch[start:start+64]
        with torch.no_grad(), autocast('cuda', dtype=torch.bfloat16):
            all_img_f.append(F.normalize(model.encode_image(ib), dim=-1).cpu())
            all_txt_f.append(F.normalize(model.encode_text(tb), dim=-1).cpu())

    img_f = torch.cat(all_img_f, dim=0).float()
    txt_f = torch.cat(all_txt_f, dim=0).float()

    matched = (img_f * txt_f).sum(dim=-1).numpy()
    sim_matrix = (img_f @ txt_f.t())
    mask = ~torch.eye(n, dtype=torch.bool)
    unmatched = sim_matrix[mask].numpy()

    gap = matched.mean() - unmatched.mean()

    ax.hist(matched, bins=50, alpha=0.7, color='forestgreen', density=True,
            label=f'matched (μ={matched.mean():.3f})')
    ax.hist(unmatched[:5000], bins=50, alpha=0.5, color='salmon', density=True,
            label=f'unmatched (μ={unmatched.mean():.3f})')
    ax.axvline(matched.mean(), color='green', linestyle='--', alpha=0.7)
    ax.axvline(unmatched.mean(), color='red', linestyle='--', alpha=0.7)
    ax.legend(fontsize=8)
    ax.set_xlabel('Cosine Similarity')
    ax.set_title(f'{name}\ngap = {gap:.3f}', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Density')
fig.suptitle('Embedding Alignment: Matched vs Unmatched Pair Similarity',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. The numbers

Summary table of everything we've measured.

In [ ]:
print('=' * 75)
print(f'{"Metric":<20} {"From Scratch":>15} {"Pretrained":>15} {"Fine-tuned":>15}')
print('=' * 75)

names = list(all_results.keys())
for metric, label in [('i2t_recall@1', 'I→T Recall@1'),
                       ('t2i_recall@1', 'T→I Recall@1'),
                       ('i2t_recall@5', 'I→T Recall@5'),
                       ('t2i_recall@5', 'T→I Recall@5')]:
    vals = [all_results[n][metric] for n in names]
    best = max(vals)
    row = f'{label:<20}'
    for v in vals:
        marker = ' ★' if v == best else '  '
        row += f'{v:>13.1%}{marker}'
    print(row)

print('=' * 75)
print(f'{"Training time":<20} {"~160 min":>15} {"0 min":>15} {"~40 min":>15}')
print(f'{"Training data seen":<20} {"1M pairs":>15} {"400M pairs":>15} {"400M + 1M":>15}')
print(f'{"Training steps":<20} {"20,000":>15} {"0":>15} {"5,000":>15}')
print('=' * 75)